<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

In [1]:
%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt

# Kalman filter

The Kalman filter (KF) is one of the most widely used state estimation algorithms in robotics applications. In the real world, we may already know the dynamics/kinematics of the robot that allows us to know how the state of the robot evolves over time given the initial state. Unfortunately, we always have to deal with imperfect (i.e., noisy) models for various reasons (e.g., imperfect parts, etc.).

Robots typically have various sensors, and we can use these sensors to also help in estimating the state of the robot. However, sensors are also not perfect. Can we combine the data from our sensors with other models that we already have to better estimate the robot state? This is where KF can help us. If we have new data about the state of the robot, together with a measure of how informative it is (for example an associated _covariance_), we can use this new information and combine it with what we already have to make a better estimate of our state. 

In KF, we assume:
1. Linear process with additive Gaussian noise $\mathbf{x}_{t} = A_t \mathbf{x}_{t-1} + B_t \mathbf{u}_{t} + \mathbf{w}_{t}$ where $\mathbf{w}_{t} \sim \mathcal{N}(0, Q_t)$
2. Linear measurement model with additive Gaussian noise $\mathbf{z}_t = H_t \mathbf{x}_t + \mathbf{n}_t$ where $\mathbf{n}_t \sim \mathcal{N}(0, R_t)$
3. Gaussian prior $bel(\mathbf{x}_0) = \mathcal{N}(\mu_0, \Sigma_0)$

There are two main steps that we need to perform: predict and update.

### Spiegazione

Una **matrice di covarianza** (indicata spesso con $\Sigma$, $Q$ o $R$ nel Filtro di Kalman) è una tabella quadrata di numeri che descrive **quanto è grande l'incertezza** su un gruppo di variabili e **come queste variabili sono collegate tra loro**.

Immaginando di dover stimare la posizione del Duckiebot sul pavimento tramite due coordinate $(x, y)$, la matrice di covarianza ha questa forma:

$$\Sigma = \begin{bmatrix} \sigma_x^2 & \text{cov}(x, y) \\ \text{cov}(y, x) & \sigma_y^2 \end{bmatrix}$$

Ogni elemento della matrice ha un significato fisico preciso:

* **La diagonale principale ($\sigma_x^2$ e $\sigma_y^2$) – La Varianza:** Indica l'incertezza della singola variabile presa da sola. Se $\sigma_x^2$ è molto grande, significa che non siamo affatto sicuri della posizione lungo l'asse $x$; se è vicina a zero, conosciamo $x$ con estrema precisione.
* **Gli elementi fuori diagonale ($\text{cov}(x, y)$) – La Covarianza:** Indica se e come gli errori delle due variabili si influenzano a vicenda (la matrice è sempre simmetrica, quindi $\text{cov}(x, y) = \text{cov}(y, x)$):
* **Covarianza $= 0$:** Le due variabili sono indipendenti. Sbagliare la stima di $x$ non dà alcuna informazione su un eventuale errore in $y$.
* **Covarianza $> 0$ (positiva):** Quando $x$ cresce (o ha un errore positivo), tende a crescere anche $y$.
* **Covarianza $< 0$ (negativa):** Quando $x$ cresce, $y$ tende a diminuire.



**L'intuizione geometrica: l'ellisse di incertezza**
Nel mondo della robotica, una matrice di covarianza in 2D corrisponde visivamente a un'**ellisse** disegnata attorno alla posizione stimata del robot:

* La **larghezza e l'altezza** dell'ellisse dipendono dai valori sulla diagonale (le varianze). Più il robot è incerto, più l'ellisse si gonfia.
* L'**inclinazione** dell'ellisse dipende dai valori fuori diagonale (le covarianze). Ad esempio, se il Duckiebot avanza in diagonale a $45^\circ$, l'incertezza dovuta allo slittamento delle ruote allungherà l'ellisse proprio lungo quella diagonale, creando una covarianza positiva tra $x$ e $y$.

Nel contesto delle equazioni del Filtro di Kalman, userai tre matrici di covarianza distinte: $\Sigma_0$ per l'incertezza iniziale, $Q_t$ per l'incertezza introdotta dal movimento delle ruote e $R_t$ per l'incertezza dei sensori (come la telecamera).



I tre assunti costituiscono le fondamenta matematiche del **Filtro di Kalman**. Servono a garantire una proprietà fondamentale: se parti da un'incertezza rappresentata da una campana di Gauss (distribuzione normale) e applichi solo trasformazioni lineari e rumori gaussiani, il risultato rimane sempre una distribuzione gaussiana. Questo permette al robot di stimare la propria posizione in tempo reale usando solo semplici operazioni tra matrici.



### 1. Modello di Processo (o di Movimento)

$$\mathbf{x}_{t} = A_t \mathbf{x}_{t-1} + B_t \mathbf{u}_{t} + \mathbf{w}_{t} \quad \text{dove} \quad \mathbf{w}_{t} \sim \mathcal{N}(0, Q_t)$$

Questa equazione descrive **come evolve lo stato del robot da un istante all'altro** in base alla fisica e ai comandi inviati ai motori (la fase di *Predizione*).

* **$\mathbf{x}_t$ e $\mathbf{x}_{t-1}$ (Vettore di stato):** Rappresenta ciò che vogliamo stimare all'istante attuale $t$ e all'istante precedente $t-1$. Nel caso del Duckiebot, ad esempio, è la sua posa sul piano: $\mathbf{x}_t = [x, y, \theta]^T$ (coordinate e orientamento).
* **$A_t$ (Matrice di transizione di stato):** Indica come lo stato precedente $\mathbf{x}_{t-1}$ si trasforma naturalmente nel nuovo stato $\mathbf{x}_t$ anche senza comandi esterni (ad esempio, se il robot ha una certa velocità, continuerà a spostarsi per inerzia).
* **$\mathbf{u}_t$ (Vettore di controllo):** È l'input che diamo al sistema, cioè i comandi inviati al robot (es. velocità lineari/angolari o segnali PWM alle ruote).
* **$B_t$ (Matrice di controllo):** Traduce i comandi $\mathbf{u}_t$ in una variazione effettiva dello stato (es. converte i giri delle ruote in metri percorsi lungo $x$ e $y$).
* **$\mathbf{w}_t$ (Rumore di processo):** Rappresenta l'imprevedibilità del mondo reale (slittamento delle ruote, attrito, irregolarità del pavimento).
* **$\mathbf{w}_t \sim \mathcal{N}(0, Q_t)$:** Significa che il rumore $\mathbf{w}_t$ segue una distribuzione gaussiana con media $0$ (non c'è un errore sistematico fisso) e **matrice di covarianza $Q_t$**. Più $Q_t$ ha valori grandi, meno ci fidiamo del modello di movimento (es. pavimento molto scivoloso).



### 2. Modello di Misura (o di Osservazione)

$$\mathbf{z}_t = H_t \mathbf{x}_t + \mathbf{n}_t \quad \text{dove} \quad \mathbf{n}_t \sim \mathcal{N}(0, R_t)$$

Questa equazione collega **lo stato reale del robot con ciò che leggono i suoi sensori** (la fase di *Aggiornamento* o *Correzione*).

* **$\mathbf{z}_t$ (Vettore di misura):** Sono i dati grezzi restituiti dai sensori all'istante $t$ (nel Duckiebot, ad esempio, la posizione rilevata tramite la telecamera leggendo un AprilTag o le linee della corsia).
* **$H_t$ (Matrice di osservazione):** Proietta lo stato reale $\mathbf{x}_t$ nello "spazio dei sensori". Serve a convertire lo stato nelle stesse unità di misura o variabili che il sensore è in grado di leggere (se lo stato include posizione e velocità, ma il sensore misura solo la posizione, $H_t$ estrae solo la posizione).
* **$\mathbf{n}_t$ (Rumore di misura):** È l'errore intrinseco del sensore (sfocatura della telecamera, riflessi di luce, risoluzione finita).
* **$\mathbf{n}_t \sim \mathcal{N}(0, R_t)$:** Indica che l'errore del sensore è gaussiano con media $0$ e **matrice di covarianza $R_t$**. Valori alti in $R_t$ indicano sensori molto rumorosi e poco affidabili; valori bassi indicano sensori molto precisi.



### 3. Distribuzione a Priori (Stato Iniziale)

$$bel(\mathbf{x}_0) = \mathcal{N}(\mu_0, \Sigma_0)$$

Per iniziare a calcolare la posizione, il filtro ha bisogno di un punto di partenza all'istante $t = 0$.

* **$bel(\mathbf{x}_0)$ (*Belief* iniziale):** È la nostra "credenza" probabilistica su dove si trovi il robot prima ancora di iniziare a muoversi o fare misure.
* **$\mu_0$ (Media iniziale):** È la nostra migliore stima iniziale dello stato (ad esempio $[0, 0, 0]^T$ se sappiamo che il robot parte dall'origine della mappa).
* **$\Sigma_0$ (Matrice di covarianza iniziale):** Misura quanto siamo sicuri di $\mu_0$. Se posiamo il robot esattamente sul punto di partenza con un righello, $\Sigma_0$ avrà valori piccolissimi; se non sappiamo affatto dove sia stato posizionato nella mappa, $\Sigma_0$ avrà valori molto grandi.

---

## Prediction step 

In the predict step we want to an obtain an estimate of $\bar{bel}(\mathbf{x}_{t}) = \mathcal{N}(\bar{\mu}_{t}, \bar{\Sigma}_{t}) $. We do this with two equations, one for our mean vector and one for our covariance matrix:

1. $\bar{\mu}_{t} = A_t\mu_{t-1} + B_tu_{t}$
2. $\bar{\Sigma}_{t} = A_t\Sigma_{t-1}A^T + Q_t$

### Spiegazione 

Il **Prediction step** (fase di predizione) è il momento in cui il Duckiebot stima dove si trova **basandosi esclusivamente sul proprio modello fisico e sui comandi inviati alle ruote**, senza ancora guardare i sensori (come la telecamera). È l'equivalente di camminare per qualche passo a occhi chiusi sapendo quanto forte stai spingendo le gambe.

Il risultato è una distribuzione gaussiana chiamata **credenza a priori**, indicata con $\bar{bel}(\mathbf{x}_{t}) = \mathcal{N}(\bar{\mu}_{t}, \bar{\Sigma}_{t})$. La **barra sopra i simboli** ($\bar{\mu}$, $\bar{\Sigma}$) serve proprio a distinguere questa stima *provvisoria* (calcolata prima della lettura dei sensori) da quella *definitiva* che otterrai nella fase successiva.



### 1. Predizione della posizione media ($\bar{\mu}_t$)

$$\bar{\mu}_{t} = A_t\mu_{t-1} + B_tu_{t}$$

Questa equazione calcola il **nuovo centro della gaussiana**, ovvero la posizione più probabile in cui si trova il robot all'istante $t$:

* **$A_t\mu_{t-1}$ (Evoluzione naturale):** Prende la migliore stima della posizione all'istante precedente ($\mu_{t-1}$) e la aggiorna secondo la fisica interna del sistema ($A_t$). Se il robot aveva già una certa velocità o inerzia, questo termine calcola dove sarebbe arrivato anche senza nuovi comandi.
* **$B_tu_{t}$ (Effetto del comando):** Prende il comando attuale inviato ai motori ($u_t$, ad esempio la velocità delle ruote) e lo converte in uno spostamento effettivo tramite la matrice $B_t$.
* **Perché non c'è il rumore $\mathbf{w}_t$?** Nell'equazione fisica vista in precedenza c'era il termine $+\mathbf{w}_t$, ma poiché abbiamo assunto che il rumore abbia **media zero** ($\mathcal{N}(0, Q_t)$), in media gli slittamenti a destra e a sinistra si compensano e non spostano il valore atteso.



### 2. Predizione dell'incertezza ($\bar{\Sigma}_t$)

$$\bar{\Sigma}_{t} = A_t\Sigma_{t-1}A_t^T + Q_t$$

Muoversi "a occhi chiusi" aumenta inevitabilmente i dubbi sulla propria posizione esatta. Questa equazione calcola quanto si è allargata l'ellisse di incertezza:

* **$A_t\Sigma_{t-1}A_t^T$ (Propagazione della vecchia incertezza):** Prende l'incertezza che avevamo al passo precedente ($\Sigma_{t-1}$) e la trasforma in base al movimento effettuato ($A_t$). Moltiplicare a destra per la trasposta $A_t^T$ è la regola dell'algebra lineare per applicare una trasformazione lineare a una matrice di covarianza. Ad esempio, se avevi un piccolo errore sull'angolo di orientamento $\theta$ e avanzi dritto per 1 metro, quell'errore angolare si traduce in un errore molto più grande sulla posizione $(x, y)$.
* **$+ Q_t$ (Aggiunta di nuova incertezza):** Somma la covarianza del rumore di processo ($Q_t$). Ogni volta che il robot si muove, le ruote possono slittare o incontrare attriti imprevisti: aggiungere $Q_t$ "gonfia" ulteriormente la matrice di covarianza.



### Cosa succede in pratica sul Duckiebot

Se il Duckiebot continuasse a eseguire solo il *Prediction step* in loop senza mai fare l'*Update* con la telecamera:

1. La media $\bar{\mu}_t$ seguirebbe la traiettoria ideale calcolata dall'odometria delle ruote, accumulando via via errori reali (deriva).
2. La matrice $\bar{\Sigma}_t$ diventerebbe sempre più grande ad ogni iterazione (perché si continua a sommare $+ Q_t$), trasformando l'ellisse di incertezza in un cerchio enorme che copre l'intera stanza.

---


## Update step

In the update step we want to obtain an estimate of $bel(\mathbf{x}_t) = \mathcal{N}(\mu_t,\Sigma_t)$. In this case we do the following steps:

1. Get measurement $\mathbf{z}_{t}$
2. Compute mean and covariance of the prediction residual $\delta_{\mu} = \mathbf{z}_{t} - H_t\bar{\mu}_{t}$ and $\delta_{\Sigma} = H_t \bar{\Sigma}_{t} H_t^T + R_t$
3. Compute Kalman gain $K_{t} = \bar{\Sigma}_{t} H_t^T \delta_{\Sigma}^{-1}$
4. Compute mean and covariance of the belief $\mu_{t} = \bar{\mu}_{t} + K_t\delta_{\mu}$ and $\Sigma_{t} = \bar{\Sigma}_{t} - K_tH_t\bar{\Sigma}_{t}$


### Spiegazione

L'**Update step** (fase di aggiornamento o correzione) è il momento in cui il Duckiebot "apre gli occhi": confronta la posizione che aveva stimato alla cieca nel *Prediction step* ($\bar{\mu}_t, \bar{\Sigma}_t$) con ciò che rilevano effettivamente i suoi sensori (ad esempio la telecamera che inquadra un AprilTag o le linee della strada).

L'obiettivo è fondere queste due fonti di informazione — entrambe affette da incertezza — per ottenere la stima definitiva e più precisa possibile dello stato attuale: $bel(\mathbf{x}_t) = \mathcal{N}(\mu_t, \Sigma_t)$ (senza più la barra sopra).



### 1. Acquisizione della misura ($\mathbf{z}_t$)

Il robot legge il dato grezzo proveniente dai sensori all'istante $t$. Ad esempio, la telecamera rileva un cartello stradale e calcola che il Duckiebot si trova a distanza $(x_{\text{cam}}, y_{\text{cam}})$.



### 2. Calcolo del Residuo di Predizione (o *Innovazione*)

$$\delta_{\mu} = \mathbf{z}_{t} - H_t\bar{\mu}_{t}$$

$$\delta_{\Sigma} = H_t \bar{\Sigma}_{t} H_t^T + R_t$$

Qui il filtro misura la **sorpresa**, cioè quanto la realtà vista dal sensore si discosta da ciò che il modello fisico si aspettava:

* **Media del residuo ($\delta_{\mu}$):** Prima trasforma la posizione predetta $\bar{\mu}_t$ nello spazio dei sensori moltiplicandola per $H_t$, poi la sottrae alla misura reale $\mathbf{z}_t$.
* Se $\delta_{\mu} = 0$, il sensore conferma esattamente la predizione del movimento: non c'è nulla da correggere.
* Se $\delta_{\mu} \neq 0$, c'è una discrepanza dovuta allo slittamento delle ruote o al rumore del sensore.


* **Covarianza del residuo ($\delta_{\Sigma}$):** Rappresenta l'incertezza totale su questo confronto. Somma due fonti di errore: l'incertezza della predizione proiettata sui sensori ($H_t \bar{\Sigma}_{t} H_t^T$) e il rumore intrinseco del sensore stesso ($R_t$).



### 3. Calcolo del Guadagno di Kalman (*Kalman Gain* $K_t$)

$$K_{t} = \bar{\Sigma}_{t} H_t^T \delta_{\Sigma}^{-1} = \frac{\bar{\Sigma}_{t} H_t^T}{H_t \bar{\Sigma}_{t} H_t^T + R_t}$$

Il **Kalman Gain ($K_t$)** è il cuore dell'algoritmo: è una matrice di pesi (concettualmente un numero compreso tra $0$ e $1$) che decide **di chi fidarsi di più** tra il modello di movimento e il sensore.

Per capirlo intuitivamente, guarda i due casi limite nella frazione:

* **Sensore molto preciso ($R_t \to 0$):** Il denominatore si riduce a $H_t \bar{\Sigma}_t H_t^T$, quindi $K_t$ diventa grande (vicino a $H_t^{-1}$). Il filtro si fida ciecamente del sensore e ignora la predizione delle ruote.
* **Sensore molto rumoroso ($R_t \to \infty$):** Il denominatore esplode, quindi $K_t \to 0$. Il filtro ignora la lettura impazzita del sensore e mantiene la traiettoria stimata dal movimento.



### 4. Aggiornamento del Belief ($\mu_t$ e $\Sigma_t$)

$$\mu_{t} = \bar{\mu}_{t} + K_t\delta_{\mu}$$

$$\Sigma_{t} = \bar{\Sigma}_{t} - K_tH_t\bar{\Sigma}_{t}$$

Infine, il filtro calcola la media e la covarianza definitive all'istante $t$:

* **Nuova posizione stimata ($\mu_t$):** Parte dalla posizione predetta $\bar{\mu}_t$ e aggiunge una frazione dell'errore $\delta_{\mu}$ pesata tramite $K_t$. In pratica sposta il centro della campana di Gauss verso la misura del sensore, fermandosi a metà strada in proporzione all'affidabilità dei due.
* **Nuova incertezza ($\Sigma_t$):** Prende l'incertezza predetta $\bar{\Sigma}_t$ e **sottrae** la quantità $K_t H_t \bar{\Sigma}_t$. Poiché abbiamo acquisito nuova informazione dall'ambiente esterno tramite un sensore, la nostra incertezza diminuisce sempre: l'ellisse attorno al Duckiebot si sgonfia e torna compatta.

Al passo temporale successivo ($t+1$), questa coppia $(\mu_t, \Sigma_t)$ diventerà il nuovo punto di partenza per il *Prediction step*, chiudendo il ciclo continuo del filtro.

---


## Example: estimating robot position with KF 

Consider robot moving in a room without obstacles. The robot is equipped with two sensors to measure distance between the robot and the walls, which allows the robot to measure the location of the robot (i.e., $x$ and $y$ positions) in the room. These sensors are not perfect, however the manufacturer provides us with the information that tells us how inaccurate these sensors are.

So, say the state of the robot is its $x$ and $y$ position in the room, and the control inputs are the velocity in each direction $v_x$ and $v_y$. The robot is initialized at $(x,y) = (0,0)$, and moves by applying constant control inputs $v_x = v_y = 1$ for 10 time steps. At each time step, after applying a control signal, the robot can take a measurement using the sensors to have an idea where the it currently is. For the sake of simplicity, assume the sensors directly return the measurement of the $(x,y)$ location in the room.

Given:

$$
A_t = A =
\begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
B_t = B = 
\begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
Q_t = Q = 
\begin{bmatrix}
0.3 & 0 \\
0 & 0.3
\end{bmatrix}
$$

$$
H_t = H = 
\begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
R_t = R = 
\begin{bmatrix}
0.75 & 0.0 \\
0.0 & 0.6
\end{bmatrix}
$$

Let's use a Kalman filter to estimate of where the robot is.

**Understanding the problem**

To understand the problem, let us plot the ideal trajectory and some possible measurements that we would get using the available sensors according to their specifications.

In [ ]:
# state = [x_pos, y_pos]
num_data = 20
ground_truth_x = np.linspace(0, 10, num=num_data + 1)
ground_truth_y = ground_truth_x.copy() # x = y

# Simulate dynamics
x_0, y_0 = 0, 0
xs, ys = [0], [0]
dynamics_noise_x_var = 0.3
dynamics_noise_y_var = 0.3
for _ in range(10):
    v_x, v_y = 1.0, 1.0
    noise_x = np.random.normal(loc=0.0, scale=dynamics_noise_x_var)
    noise_y = np.random.normal(loc=0.0, scale=dynamics_noise_y_var)
    new_x = xs[-1] + v_x + noise_x
    new_y = ys[-1] + v_y + noise_y
    xs.append(new_x)
    ys.append(new_y)
    
# Simulate measurements
measurement_noise_x_var = 0.75
measurement_noise_y_var = 0.6
noise_x = np.random.normal(loc=0.0, scale=measurement_noise_x_var, size=num_data-1)
noise_y = np.random.normal(loc=0.0, scale=measurement_noise_y_var, size=num_data-1)
measurement_x = np.linspace(1, 10, num=num_data-1) + noise_x
measurement_y = np.linspace(1, 10, num=num_data-1) + noise_y

# Compare ground truth and measurements
plt.figure(figsize=(8,8))
plt.plot(ground_truth_x, ground_truth_y)
plt.plot(measurement_x, measurement_y)
plt.plot(xs, ys)
plt.xlabel('x position')
plt.ylabel('y position')
plt.legend(['Actual robot trajectory', 'Estimate from measurements alone', 'Estimate from model predictions alone'])
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

The above plot shows the ground truth robot trajectory, as well as the estimate we get from the measurements, and the estimate that we get from the integrating forward the process model (known as dead reckoning). Our objective with the Kalman filter is to get a better estimate by fusing the dynamics estimate and measurements together. 

### Predict Step

Recall the prediction step (i.e., getting $\bar{bel}(\mathbf{x}_{t}) = \mathcal{N}(\bar{\mu}_{t}, \bar{\Sigma}_{t}) $):
1. $\bar{\mu}_{t} = A\mu_{t-1} + Bu_{t}$
2. $\bar{\Sigma}_{t} = A\Sigma_{t-1}A^T + Q$

In [ ]:
# To multiply a 2D matrix and a 1D vector, use 'np.dot()' and not the '*' operator
def predict(A, B, Q, mu_t, u_t, Sigma_t):
    predicted_mu = mu_t  # TODO Need to update
    predicted_Sigma = Sigma_t # TODO Need to update
    return predicted_mu, predicted_Sigma

### Update step

Recall the update step:
1. Get measurement $\mathbf{z}_{t}$
2. Compute mean and covariance of the prediction residual $\delta_{\mu} = \mathbf{z}_{t} - H\bar{\mu}_{t}$ and $\delta_{\Sigma} = H \bar{\Sigma}_{t} H^T + R$
3. Compute Kalman gain $K_{t} = \bar{\Sigma}_{t} H^T \delta_{\Sigma}^{-1}$
4. Compute mean and covariance of the belief $\mu_{t} = \bar{\mu}_{t} + K\delta_{\mu}$ and $\Sigma_{t} = \bar{\Sigma}_{t} - KH\bar{\Sigma}_{t}$

In [ ]:
# To multiply a 2D matrix and a 1D vector, use 'np.dot()' and not the '*' operator
def update(H, R, z, predicted_mu, predicted_Sigma):
    updated_mu = predicted_mu # TODO Need to update
    updated_Sigma = predicted_Sigma # TODO Need to update
    return updated_mu, updated_Sigma

Let's run our Kalman filter!

In [ ]:
# Initialize the problem
mu_0 = np.array([0, 0])
Sigma_0 = np.array([[0.1, 0],
                     [0, 0.1]]) # We're pretty certain about bel_0
A = np.array([[1, 0],
              [0, 1]])
B = np.array([[1, 0],
              [0, 1]])
Q = np.array([[dynamics_noise_x_var, 0],
              [0, dynamics_noise_x_var]])
H = np.array([[1, 0],
              [0, 1]])
R = np.array([[measurement_noise_x_var, 0],
              [0, measurement_noise_y_var]])

# Initialize empty lists for mus and measurements for plotting
measurements = []
filtered_mus = []

# Run KF for each time step
mu_current = mu_0.copy()
Sigma_current = Sigma_0.copy()
for i in range(num_data-1):
    u_t = np.array([1, 1])
    
    # Predict step
    predicted_mu, predicted_Sigma = predict(A, B, Q, 
                                            mu_current, u_t, 
                                            Sigma_current)
    
    # Get measurement (irl, get this from our sensor)
    measurement_noise_x = np.random.normal(loc=0.0, scale=measurement_noise_x_var)
    measurement_noise_y = np.random.normal(loc=0.0, scale=measurement_noise_y_var)
    measurement_x_new = ground_truth_x[i+1] + measurement_noise_x
    measurement_y_new = ground_truth_x[i+1] + measurement_noise_y
    z = np.array([measurement_x_new, measurement_y_new])
    
    # The rest of update step
    mu_current, Sigma_current = update(H, R, z, 
                                   predicted_mu, 
                                   predicted_Sigma)
    
    # Store measurements and mu_current so we can plot it later
    measurements.append([measurement_x_new, measurement_y_new])
    filtered_mus.append(mu_current)

# Just for plotting purposes, convert the lists to array 
measurements = np.array(measurements)
filtered_mus = np.array(filtered_mus) 

In [ ]:
# Let's plot the results

plt.figure(figsize=(8,8))
plt.plot(ground_truth_x, ground_truth_y)
plt.plot(measurements[:,0], measurements[:,1])
plt.plot(xs, ys)
plt.plot(filtered_mus[:,0], filtered_mus[:,1])
plt.xlabel('x position')
plt.ylabel('y position')
plt.legend(['Actual robot trajectory', 'Estimate from measurements alone', 'Estimate from dynamics alone', 'Estimate fused from Kalman Filter'])
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

You should  see that the KF is able to fuse all of the information together and give us a better final estimate of the robot trajectory.

Now move on the next notebook about [particle filters](../02-particle-filter/particle-filter.ipynb)